[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-random-forest.ipynb)

# Random Forest

*AIBits Academy · Machine Learning End To End · Ensemble Learning*

An ensemble of decision trees trained on random bootstrap samples with random feature subsets — combining many weak learners into a single powerful, robust model.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> A single decision tree is a lot like asking one expert their opinion — brilliant when they're right, catastrophically wrong when they're not. Random Forest is the **wisdom of the crowd** applied to trees: instead of one deeply-grown tree memorising every quirk of the training data, grow a hundred medium-depth trees, deliberately show each of them a slightly different slice of the data (bootstrap sampling), let each split from a slightly different set of features (feature bagging) — and then take a majority vote. Any individual tree in the forest might be over-confident about an outlier or fooled by a coincidental correlation, but averaged across 100 differently-informed trees, the idiosyncratic mistakes cancel out and the shared signal survives. That's why a forest almost always beats its own component trees.

> **📋 Real-World Case Study — Credit Card Lead Prediction (Bank Cross-Sell)**
>
> A bank scored 245,725 existing customers for likelihood of taking up a credit-card offer (23.7% positive base rate) using Logistic Regression, Decision Tree, Random Forest, and AdaBoost. The default Random Forest landed at 77.5% test accuracy — respectable, but *lower* than a properly depth-limited Decision Tree (78.6%, rising to 79.1% after a small GridSearchCV over `max_depth`/`max_leaf_nodes`/`min_samples_split`). A wider grid search over the Random Forest actually made things worse (76.2% best CV score) by forcing `min_impurity_decrease` too high, starving every tree of splits. The lesson generalises: bagging reduces variance, but it is not a free lunch — an ensemble with a badly-chosen search space can lose to a single well-tuned tree, so the hyperparameter grid deserves as much attention as the model family.

## Two Sources of Randomness

**1. Bootstrap Aggregating (Bagging):** Each tree is trained on a random sample with replacement (bootstrap) of the training data. About 63.2% of samples are used; the rest (~36.8%) form the Out-of-Bag (OOB) set for free validation. 
 
**2. Random Feature Subsets:** At each split, only a random subset of √p (classification) or p/3 (regression) features are considered. This decorrelates the trees — they disagree and compensate for each other's errors. 
 
**Prediction:** Classification → majority vote across all trees. Regression → average of all trees.

## Why Does Averaging Trees Help?

$$\text{Var}(\text{mean of } n \text{ trees}) = \frac{\sigma^2}{n}\big[1+(n-1)\rho\big]$$

Where ρ is the pairwise correlation between trees. If trees are independent (ρ≈0), variance drops as 1/n. Random feature selection actively reduces ρ, making the forest far better than any individual tree.

## Try It — Watch Var(mean) Drop as Trees Are Added

Each dot is one tree's prediction (noisy, scattered around the true value). Drag n and ρ and watch the ensemble average — the bold line — stabilise. The readout below computes Var(mean) two ways: from the formula above, and empirically from 300 freshly-simulated forests at the current settings, confirming they agree.

## From Scratch (using our DT)

In [ ]:
import numpy as np
from collections import Counter
from sklearn.tree import DecisionTreeClassifier

class RandomForestClassifier:
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt'):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.max_features=max_features
        self.trees_=[]; self.feature_indices_=[]

    def fit(self, X, y):
        X,y=np.array(X),np.array(y)
        n,p=X.shape
        nf=(int(np.sqrt(p)) if self.max_features=='sqrt' else max(1,p//3))
        for _ in range(self.n_estimators):
            idx=np.random.choice(n, n, replace=True)    # bootstrap
            feat_idx=np.random.choice(p, nf, replace=False)  # feature subset
            tree=DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X[idx][:,feat_idx], y[idx])
            self.trees_.append(tree); self.feature_indices_.append(feat_idx)

    def predict(self, X):
        X=np.array(X)
        all_preds=np.array([t.predict(X[:,fi]) for t,fi
                            in zip(self.trees_, self.feature_indices_)])
        return np.array([Counter(all_preds[:,i]).most_common(1)[0][0]
                         for i in range(X.shape[0])])

# Zomato restaurant survival prediction
np.random.seed(42)
n=600
X=np.column_stack([np.random.uniform(1,5,n),    # avg rating
                   np.random.randint(50,2000,n),  # num reviews
                   np.random.randint(1,8,n),       # locality score
                   np.random.uniform(100,1500,n)]) # avg order ₹
y=((X[:,0]>3.5)&(X[:,1]>300)).astype(int)

from sklearn.model_selection import train_test_split
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,random_state=0)

rf=RandomForestClassifier(n_estimators=100, max_depth=6)
rf.fit(X_tr, y_tr)
print(f"Scratch RF accuracy: {np.mean(rf.predict(X_te)==y_te):.3f}")

from sklearn.ensemble import RandomForestClassifier as SKRF
skrf=SKRF(n_estimators=100, max_depth=6, random_state=42)
skrf.fit(X_tr,y_tr)
print(f"sklearn  RF accuracy: {skrf.score(X_te,y_te):.3f}")
feat_names=['Rating','Reviews','Locality','Avg Order ₹']
for n,imp in sorted(zip(feat_names, skrf.feature_importances_), key=lambda x:-x[1]):
    print(f"  {n}: {imp:.3f}")

## Reading the Feature-Importance Output — Which Features Actually Drive Predictions?

The four `feature_importances_` values from the code above sum to 1.0 by construction (that's how sklearn's Mean Decrease in Impurity is normalised). Plotted as a bar chart, they immediately reveal the ranking that a raw print-out obscures:

In this Zomato-survival toy dataset, the labelling rule was `y = (rating > 3.5) & (reviews > 300)` — only two features actually determine the label, so it's exactly right that Rating dominates (0.832) and Reviews takes almost all the rest (0.150). Locality and Avg Order value together carry < 2% importance — the forest has correctly learned that they are noise features here. This is precisely the diagnostic value of the bar chart: on a real-world Flipkart or Swiggy problem where you don't know the ground-truth relationships, a chart like this tells you which columns your model is actually leaning on — and whether removing the tiny-importance ones would simplify the pipeline with no accuracy loss. Remember the standard caveats from the Q&A below: MDI is biased toward high-cardinality features, and correlated features split each other's importance — for high-stakes attribution, cross-check with permutation importance or SHAP.

## Key Hyperparameters

| Parameter | Default | Effect | Tune to |
|---|---|---|---|
| n_estimators | 100 | More trees → lower variance, more compute | Start 100, try 200–500 |
| max_features | 'sqrt' | Lower → more diverse trees (less correlation) | 'sqrt' for classification, 1/3 for regression |
| max_depth | None | Shallower → lower variance, less overfitting | 5–20 via cross-validation |
| min_samples_leaf | 1 | Higher → smoother decision boundary | 2–10 for noisy data |
| oob_score | False | Free validation estimate | Set True as a sanity check |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A forest and its out-of-bag score

Fit a scikit-learn `RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0)` on the moons data (note the name: it is sklearn's, not the one defined earlier in the lesson). Store the OOB accuracy in `oob`.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
X, y = make_moons(n_samples=500, noise=0.25, random_state=0)
oob = None   # TODO


In [ ]:
try:
    check("OOB accuracy is high", oob > 0.85)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
X, y = make_moons(n_samples=500, noise=0.25, random_state=0)
oob = SkRandomForest(n_estimators=200, oob_score=True, random_state=0).fit(X, y).oob_score_

```

</details>

### Exercise 2 · Medium · Which feature matters?

Only column 0 drives the label. Fit a forest and store the index of the most important feature in `top_feature` and its importance in `top_importance`.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
rng = np.random.default_rng(0)
X = rng.normal(size=(500, 5))
y = (X[:, 0] + 0.2 * rng.normal(size=500) > 0).astype(int)
top_feature = top_importance = None   # TODO


In [ ]:
try:
    check("column 0 wins", top_feature == 0)
    check("with most of the importance", top_importance > 0.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
rng = np.random.default_rng(0)
X = rng.normal(size=(500, 5))
y = (X[:, 0] + 0.2 * rng.normal(size=500) > 0).astype(int)
imp = SkRandomForest(n_estimators=200, random_state=0).fit(X, y).feature_importances_
top_feature = int(imp.argmax())
top_importance = float(imp.max())

```

</details>

### Exercise 3 · Stretch · Bootstrap and the 36.8% rule

Write `bootstrap_split(n, rng)` returning `(in_bag, out_of_bag)` index arrays: `in_bag` is `n` draws **with replacement**; `out_of_bag` is every index never drawn. About 36.8% (1/e) of the rows end up out of the bag.

In [ ]:
import numpy as np
def bootstrap_split(n, rng):
    pass   # TODO


In [ ]:
try:
    rng = np.random.default_rng(0)
    ib, oob = bootstrap_split(1000, rng)
    check("in-bag has n draws", len(ib) == 1000)
    check("in-bag repeats rows", len(set(ib)) < 1000)
    check("out-of-bag share near 0.368", 0.33 < len(oob) / 1000 < 0.41)
    check("disjoint", set(ib).isdisjoint(set(oob)))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def bootstrap_split(n, rng):
    in_bag = rng.integers(0, n, n)
    out_of_bag = np.setdiff1d(np.arange(n), in_bag)
    return in_bag, out_of_bag

```

Those left-out rows are a free validation set for each tree — that is the out-of-bag score.

</details>

---
*Back to the course: **Machine Learning End To End → Random Forest**.*